In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector, RFE, RFECV
from sklearn.metrics import accuracy_score

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load a Real Dataset

We use the **Breast Cancer dataset** built into scikit-learn.

For classroom demonstration, we use 10 numerical features so that the wrapper search remains easy to understand and reasonably fast.


In [96]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

Random_features = [
    "mean radius", "mean texture", "mean perimeter", "mean area",
    "mean smoothness", "mean compactness", "mean concavity",
    "mean symmetry", "radius error", "texture error"
]

X = X[Random_features]

print("Dataset shape:", X.shape)
print("Classes:", y.unique())
X.head()

Dataset shape: (569, 10)
Classes: [0 1]


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean symmetry,radius error,texture error
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.2419,1.0950,0.9053
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.1812,0.5435,0.7339
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.2069,0.7456,0.7869
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.2597,0.4956,1.1560
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.1809,0.7572,0.7813


## 3. Train/Test Split

Feature selection is learned using the **training data**. The test set is kept untouched until final evaluation.

This helps prevent **data leakage**.


In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)


Training shape: (455, 10)
Testing shape : (114, 10)


# Part A — Forward Selection

### Definition
Forward Selection starts with **no features** and adds one feature at a time.

### Steps
1. Start with an empty set.
2. Try adding each remaining feature.
3. Evaluate model performance.
4. Permanently add the feature giving the best improvement.
5. Repeat until the desired number of features is selected.

**Student definition:**  
> Start with nothing and keep adding the most useful feature.

**Search idea:** maximize `Score(S ∪ {x})`.


In [66]:
forward_selector = SequentialFeatureSelector(
    estimator=model,
    n_features_to_select=5,
    direction="forward",
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

forward_selector.fit(X_train, y_train)

forward_features = X_train.columns[forward_selector.get_support()]

print("Forward-selected features:")
for f in forward_features:
    print("-", f)


Forward-selected features:
- mean radius
- mean texture
- mean perimeter
- mean smoothness
- mean concavity


In [67]:
Xtr = forward_selector.transform(X_train)
Xte = forward_selector.transform(X_test)

forward_model = model.fit(Xtr, y_train)
forward_accuracy = accuracy_score(y_test, forward_model.predict(Xte))

print("Selected features:", len(forward_features))
print("Test accuracy:", round(forward_accuracy, 4))


Selected features: 5
Test accuracy: 0.8947


### Forward Selection Flow

`Empty Set → Try Adding → Evaluate → Add Best → Repeat`

Example:

`∅ → {C} → {C,D} → {C,D,A} → ...`


# Part B — Backward Elimination

### Definition
Backward Elimination starts with **all features** and removes one feature at a time.

### Steps
1. Start with all features.
2. Try removing each feature.
3. Evaluate model performance.
4. Permanently remove the feature whose removal is best.
5. Repeat until the desired number remains.

**Student definition:**  
> Start with everything and keep removing the least useful feature.

**Search idea:** maximize `Score(S − {x})`.


In [72]:
backward_selector = SequentialFeatureSelector(
    estimator=model,
    n_features_to_select=5,
    direction="backward",
    scoring="accuracy",
    cv=5,
    n_jobs=-1
)

backward_selector.fit(X_train, y_train)

backward_features = X_train.columns[backward_selector.get_support()]

print("Backward-selected features:")
for f in backward_features:
    print("-", f)


Backward-selected features:
- mean radius
- mean texture
- mean smoothness
- radius error
- texture error


In [73]:
Xtr = backward_selector.transform(X_train)
Xte = backward_selector.transform(X_test)

backward_model = model.fit(Xtr, y_train)
backward_accuracy = accuracy_score(y_test, backward_model.predict(Xte))

print("Selected features:", len(backward_features))
print("Test accuracy:", round(backward_accuracy, 4))


Selected features: 5
Test accuracy: 0.8947


### Backward Elimination Flow

`All Features → Try Removing → Evaluate → Remove Best Candidate → Repeat`


# Part C — Recursive Feature Elimination (RFE)

### Definition
RFE repeatedly:
1. trains a model,
2. ranks features using model coefficients/importances,
3. removes the weakest feature(s),
4. retrains,
5. repeats until the desired number remains.

**Student definition:**  
> RFE asks the model: **“Which features can I remove next?”**

### Flow

`All Features → Train Model → Rank Features → Remove Weak Features → Train Again → Repeat`


In [83]:
rfe_estimator = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])

rfe = RFE(estimator=rfe_estimator, n_features_to_select=5, step=1,importance_getter="named_steps.classifier.coef_")

rfe.fit(X_train, y_train)

rfe_features = X_train.columns[rfe.support_]

print("RFE-selected features:")
for f in rfe_features:
    print("-", f)

RFE-selected features:
- mean radius
- mean texture
- mean perimeter
- mean area
- mean smoothness


In [85]:
rfe_ranking = pd.DataFrame({
    "Feature": X_train.columns,
    "Selected": rfe.support_,
    "RFE Rank": rfe.ranking_
}).sort_values("RFE Rank")

rfe_ranking


,Feature,Selected,RFE Rank
0,mean radius,True,1
1,mean texture,True,1
2,mean perimeter,True,1
3,mean area,True,1
4,mean smoothness,True,1
6,mean concavity,False,2
7,mean symmetry,False,3
9,texture error,False,4
8,radius error,False,5
5,mean compactness,False,6


In [87]:
Xtr = rfe.transform(X_train)
Xte = rfe.transform(X_test)

rfe_model = rfe_estimator.fit(Xtr, y_train)
rfe_accuracy = accuracy_score(y_test, rfe_model.predict(Xte))

print("Selected features:", len(rfe_features))
print("Test accuracy:", round(rfe_accuracy, 4))


Selected features: 5
Test accuracy: 0.8596


## RFE vs RFECV

**RFE:** we specify how many features to keep.

**RFECV:** RFE + Cross-Validation; cross-validation helps select a suitable number of features.

Example:

```python
from sklearn.feature_selection import RFECV
```


In [91]:
# RFECV demonstration
rfecv = RFECV(
    estimator=rfe_estimator,
    step=1,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    importance_getter="named_steps.classifier.coef_"
)

rfecv.fit(X_train, y_train)

rfecv_features = X_train.columns[rfecv.support_]

print("RFECV selected number of features:", len(rfecv_features))
print("RFECV features:")
for f in rfecv_features:
    print("-", f)


RFECV selected number of features: 6
RFECV features:
- mean radius
- mean texture
- mean perimeter
- mean area
- mean smoothness
- mean concavity


# Practical Exercise

### Exercise 1
Change `n_features_to_select=5` to `3` for:
- Forward Selection
- Backward Elimination
- RFE

Compare the selected features.

### Exercise 2
Replace Logistic Regression with:

```python
from sklearn.tree import DecisionTreeClassifier
```

Then observe whether the selected features change.

### Exercise 3 — Think
1. Why can Forward and Backward select different features?
2. Why are wrapper methods computationally expensive?
3. Why must the test set remain untouched?
4. Why does RFE depend on the estimator?
5. What problem occurs when the number of features becomes very large?
